In [33]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize

In [ ]:
# ============================================================
# Portfolio and Risk Management - Unconstrained Optimization
# ============================================================

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
pd.set_option('display.width', 220)

FILE = "./spx_returns_weekly.xlsx"
PERIODS = 52        

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------
tickers_10 = ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','BRK/B','LLY']

spx = pd.read_excel(FILE, sheet_name='spx returns')
add = pd.read_excel(FILE, sheet_name='additional returns')


spx['date'] = pd.to_datetime(spx['date'])
add['date'] = pd.to_datetime(add['date'])

returns_df = (spx[['date'] + tickers_10]
              .merge(add[['date', 'SPY']], on='date', how='inner')
              .set_index('date')
              .sort_index())

returns_df = returns_df.dropna()
print("Final shape:", returns_df.shape)
print("Columns:", list(returns_df.columns))

Final shape: (573, 11)
Columns: ['AAPL', 'NVDA', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'AVGO', 'BRK/B', 'LLY', 'SPY']


## 1. Risk Statistics

### 1.1

Refer to the table.

### 1.2

**Most attractive: NVDA.** It has the highest Sharpe ratio by a wide margin, positive skewness, and low excess kurtosis. The trade-off is the highest volatility and a large drawdown, so it wins on a risk-adjusted basis, not on safety.

**Least attractive: META.** Lowest Sharpe, third-highest volatility, worst drawdown, and fat tails. It is dominated on every risk-adjusted dimension.

### 1.3

**Most attractive vs SPY: LLY.** It has a high alpha and info ratio combined with a low beta, so most of its return is not explained by the market (low R²). NVDA has the largest alpha and info ratio, but its beta is high, so much of its return is just market exposure rather than true outperformance.

---

In [35]:
# ------------------------------------------------------------
# 1. Risk Statistics
# ------------------------------------------------------------
def max_drawdown(r):
    cum = (1 + r).cumprod()
    return (cum / cum.cummax() - 1).min()

def risk_stats_table(df, periods=PERIODS):
    rows = {}
    for c in df.columns:
        r = df[c]
        mean_ann = r.mean() * periods
        vol_ann  = r.std(ddof=1) * np.sqrt(periods)
        sharpe   = mean_ann / vol_ann if vol_ann != 0 else np.nan
        rows[c] = [mean_ann, vol_ann, sharpe, r.skew(), r.kurtosis(), max_drawdown(r)]
    return pd.DataFrame(rows,
        index=['mean (ann)','vol (ann)','Sharpe (ann)',
               'skewness','kurtosis','max drawdown']).T

stats = risk_stats_table(returns_df)
print("\n=== 1.1 Risk Statistics ===")
print(stats)

# 1.2
print("\n=== 1.2 Standalone attractiveness ===")
best  = stats['Sharpe (ann)'].idxmax()
worst = stats['Sharpe (ann)'].idxmin()
print(f"Most attractive : {best}  (Sharpe={stats.loc[best,'Sharpe (ann)']:.4f})")
print(f"Least attractive: {worst} (Sharpe={stats.loc[worst,'Sharpe (ann)']:.4f})")

# 1.3 Regression vs SPY
def regression_vs_spy(df, spy='SPY'):
    X = sm.add_constant(df[spy])
    rows = {}
    for c in df.columns:
        if c == spy:
            continue
        m = sm.OLS(df[c], X).fit()
        alpha_ann = m.params['const'] * PERIODS
        beta      = m.params[spy]
        resid_std = np.std(m.resid, ddof=1)
        ir        = alpha_ann / (resid_std * np.sqrt(PERIODS)) if resid_std > 0 else np.nan
        rows[c]   = [alpha_ann, beta, ir, m.rsquared]
    return pd.DataFrame(rows,
        index=['alpha (ann)','beta','info ratio','r-squared']).T

reg = regression_vs_spy(returns_df)
print("\n=== 1.3 Regression vs SPY ===")
print(reg)


=== 1.1 Risk Statistics ===
       mean (ann)  vol (ann)  Sharpe (ann)  skewness  kurtosis  max drawdown
AAPL       0.2485     0.2776        0.8952   -0.1821    1.8047       -0.3464
NVDA       0.6453     0.4544        1.4202    0.3509    1.4917       -0.6594
MSFT       0.2353     0.2378        0.9894    0.1327    1.6850       -0.3505
GOOGL      0.2698     0.2873        0.9392    0.5326    3.1040       -0.4186
AMZN       0.2613     0.3045        0.8581   -0.0872    1.3419       -0.5483
META       0.2320     0.3575        0.6491    0.0370    3.4901       -0.7603
TSLA       0.4375     0.5797        0.7548    0.5593    1.6335       -0.7225
AVGO       0.3939     0.3825        1.0297    0.6429    3.2722       -0.4004
BRK/B      0.1348     0.1882        0.7164   -0.1865    2.5339       -0.2648
LLY        0.3014     0.2982        1.0107    0.1383    1.9761       -0.3448
SPY        0.1452     0.1690        0.8588   -0.5812    5.9670       -0.3183

=== 1.2 Standalone attractiveness ===
Most att

## 2. Portfolio Allocation

### 2.1

The most correlated pair is **MSFT–SPY**; the least correlated pair is **AVGO–LLY**.

**Expected extra weight: LLY.** It has the lowest correlations with the tech cluster, a low beta to SPY, and a top-three Sharpe ratio, so it contributes the most diversification.

### 2.2

**No, the best-Sharpe assets do not get the biggest weights.** NVDA has the highest Sharpe but only the third-largest weight, while BRK/B has one of the lowest Sharpes yet the largest long weight. SPY receives a large short position. Weights are driven by the covariance structure, not by Sharpe alone — low-correlation assets are levered up, and highly correlated assets are used as hedges or shorts.

### 2.3

Refer to the table.

### 2.4

The new weights are far more balanced: the extreme long in BRK/B and the large short in SPY disappear, and the short leg becomes small. The cost is a lower Sharpe ratio.

**What is going on?** In the unconstrained problem, SPY is shorted as a cheap hedge to finance a leveraged long portfolio of high-alpha stocks. Because SPY is highly correlated with the longs, shorting it removes common market exposure and leaves mostly residual alpha. Removing SPY forces the optimizer to express the same views using long/short positions within the stock set, which is less efficient but more realistic.

In [36]:
# ------------------------------------------------------------
# 2. Portfolio Allocation
# ------------------------------------------------------------
# 2.1 Correlation matrix
corr = returns_df.corr()
print("\n=== 2.1 Correlation matrix ===")
print(corr)

corr_pairs = corr.where(~np.eye(len(corr), dtype=bool)).stack().sort_values()
print("\nMost correlated pair:\n", corr_pairs.tail(1))
print("\nLeast correlated pair:\n", corr_pairs.head(1))

# 2.2 Tangency portfolio
def tangency_weights(df, periods=PERIODS):
    mu    = df.mean().values * periods
    Sigma = df.cov().values * periods
    n     = len(mu)

    def neg_sharpe(w):
        return -(w @ mu) / np.sqrt(w @ Sigma @ w)

    cons = ({'type': 'eq', 'fun': lambda w: w.sum() - 1})
    w0   = np.ones(n) / n
    res  = minimize(neg_sharpe, w0, constraints=cons, method='SLSQP',
                    options={'maxiter': 2000, 'ftol': 1e-12})
    return pd.Series(res.x, index=df.columns)

w_opt = tangency_weights(returns_df)
sharpe_each = stats['Sharpe (ann)'].reindex(w_opt.index)
table_22 = pd.DataFrame({'optimal weight': w_opt, 'Sharpe': sharpe_each})
print("\n=== 2.2 Tangency portfolio weights ===")
print(table_22.sort_values('optimal weight', ascending=False))

# 2.3 Portfolio performance
mu    = returns_df.mean() * PERIODS
Sigma = returns_df.cov() * PERIODS
p_mean   = w_opt.values @ mu.values
p_vol    = np.sqrt(w_opt.values @ Sigma.values @ w_opt.values)
p_sharpe = p_mean / p_vol
print("\n=== 2.3 Optimized portfolio performance ===")
print(f"Mean (ann):   {p_mean:.4f}")
print(f"Vol  (ann):   {p_vol:.4f}")
print(f"Sharpe (ann): {p_sharpe:.4f}")

# 2.4 Drop biggest short
biggest_short = w_opt.idxmin()
print(f"\n=== 2.4 Biggest short: {biggest_short} (w={w_opt.min():.4f}) ===")

df_red = returns_df.drop(columns=[biggest_short])
w_opt2 = tangency_weights(df_red)

mu2    = df_red.mean() * PERIODS
Sigma2 = df_red.cov() * PERIODS
p_mean2   = w_opt2.values @ mu2.values
p_vol2    = np.sqrt(w_opt2.values @ Sigma2.values @ w_opt2.values)
p_sharpe2 = p_mean2 / p_vol2

print("\nNew weights:")
print(w_opt2.sort_values(ascending=False))
print(f"\nNew portfolio: mean={p_mean2:.4f}, vol={p_vol2:.4f}, Sharpe={p_sharpe2:.4f}")


=== 2.1 Correlation matrix ===
        AAPL   NVDA   MSFT  GOOGL   AMZN   META   TSLA   AVGO  BRK/B    LLY    SPY
AAPL  1.0000 0.4822 0.5536 0.5380 0.4593 0.4061 0.4374 0.5070 0.4053 0.1895 0.6796
NVDA  0.4822 1.0000 0.5921 0.4514 0.5463 0.4278 0.4149 0.5726 0.2997 0.1619 0.6390
MSFT  0.5536 0.5921 1.0000 0.6069 0.6291 0.5675 0.4081 0.5329 0.3588 0.2554 0.7222
GOOGL 0.5380 0.4514 0.6069 1.0000 0.5914 0.5098 0.3664 0.4771 0.3385 0.1696 0.6494
AMZN  0.4593 0.5463 0.6291 0.5914 1.0000 0.5269 0.3954 0.4445 0.2699 0.1445 0.6142
META  0.4061 0.4278 0.5675 0.5098 0.5269 1.0000 0.2650 0.4028 0.2752 0.1159 0.5590
TSLA  0.4374 0.4149 0.4081 0.3664 0.3954 0.2650 1.0000 0.3733 0.1975 0.1258 0.5100
AVGO  0.5070 0.5726 0.5329 0.4771 0.4445 0.4028 0.3733 1.0000 0.3025 0.1045 0.6289
BRK/B 0.4053 0.2997 0.3588 0.3385 0.2699 0.2752 0.1975 0.3025 1.0000 0.2973 0.7019
LLY   0.1895 0.1619 0.2554 0.1696 0.1445 0.1159 0.1258 0.1045 0.2973 1.0000 0.3433
SPY   0.6796 0.6390 0.7222 0.6494 0.6142 0.5590 0.5100 